<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inference and Kaggle submission

This notebook does not train anything. It grabs the model we picked as the best one
from the MLflow model registry, runs it on the raw test file exactly as Kaggle gives
it to us, and writes the submission. The whole point of logging each model as a full
Pipeline was so this step needs zero manual preprocessing: the raw test frame goes in
one end and predictions come out the other.

In [ ]:
!pip -q install mlflow kaggle

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
tracking_uri = setup_env()
print("MLflow \u2192", tracking_uri)

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)

## Imports and model wrapper classes

The calendar and lag steps live in `src.features`, so they load by themselves. The two
LightGBM wrapper classes were defined inside the LightGBM notebook, so we redeclare them
here with the exact same code. cloudpickle usually embeds them in the saved model anyway,
but having them in scope makes the load safe either way. If you ever register a different
architecture as the champion (N-BEATS, DLinear, TFT), paste that notebook's wrapper class
here instead.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from src.features import CalendarFeatures, LagFeatures, DropColumns

CATEGORICAL_COLS = ["Store", "Dept"]

# these two are only needed if the champion used the log-target variant; harmless otherwise
def signed_log1p(y):
    return np.sign(y) * np.log1p(np.abs(y))

def signed_expm1(z):
    return np.sign(z) * np.expm1(np.abs(z))

def holiday_weights(is_holiday):
    return np.where(np.asarray(is_holiday) == 1, 5.0, 1.0)

def to_lgb_frame(df, train_categories=None):
    """Split a transformed frame into X (Store/Dept as category dtype), y, weights."""
    y = df["Weekly_Sales"].to_numpy(dtype=float) if "Weekly_Sales" in df.columns else None
    X = df.drop(columns=["Weekly_Sales"]).copy() if "Weekly_Sales" in df.columns else df.copy()
    w = holiday_weights(df["IsHoliday"])
    for col in CATEGORICAL_COLS:
        if train_categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=train_categories[col])
    return X, y, w

class ToLGBFrame(BaseEstimator):
    """sklearn wrapper around to_lgb_frame so categorical alignment lives in the Pipeline."""
    def fit(self, X, y=None):
        X_lgb, _, _ = to_lgb_frame(X)
        self.train_categories_ = {c: X_lgb[c].cat.categories for c in CATEGORICAL_COLS}
        return self
    def transform(self, X):
        X_lgb, _, _ = to_lgb_frame(X, train_categories=self.train_categories_)
        return X_lgb

class LGBMBoosterRegressor(BaseEstimator, RegressorMixin):
    """sklearn wrapper around a raw lgb.Booster."""
    def __init__(self, num_boost_round=100, **lgb_params):
        self.num_boost_round = num_boost_round
        self.lgb_params = lgb_params
    def fit(self, X, y, sample_weight=None):
        train_set = lgb.Dataset(X, label=y, weight=sample_weight,
                                categorical_feature=CATEGORICAL_COLS, free_raw_data=False)
        self.booster_ = lgb.train(self.lgb_params, train_set, num_boost_round=self.num_boost_round)
        return self
    def predict(self, X):
        return self.booster_.predict(X, num_iteration=self.booster_.best_iteration)

## Put the champion in the model registry

Each training notebook logs its final full-train pipeline under a run named
`final_full_train__...`. Our best validation WMAE was the LightGBM run, so we look up its
latest final run in the `lightGBM_Training` experiment and register that pipeline under a
single stable name. Running this again just adds a new version, so it is safe to rerun.

If you decide a different model is the champion, point `SOURCE_EXPERIMENT` at that model's
experiment name and rerun.

In [ ]:
MODEL_NAME        = "walmart-sales-champion"
SOURCE_EXPERIMENT = "lightGBM_Training"   # the experiment whose final_full_train run we register

client = MlflowClient()
exp = client.get_experiment_by_name(SOURCE_EXPERIMENT)
assert exp is not None, f"experiment {SOURCE_EXPERIMENT!r} not found on the tracking server"

# the most recent final_full_train__ run in that experiment is the pipeline we want
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="tags.mlflow.runName LIKE 'final_full_train__%'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)
assert runs, "no final_full_train__ run found; run the training notebook's last cell first"
champion_run = runs[0]
run_id = champion_run.info.run_id
print("champion run:", champion_run.data.tags.get("mlflow.runName"),
      "| logged val WMAE:", champion_run.data.params.get("val_wmae_of_chosen_cfg"))

# MLflow 3 stores log_model() output as a standalone "logged model" with its own id and
# artifact location, NOT under runs:/<run_id>/model (that path is empty on DagsHub). So
# mlflow.register_model("runs:/.../model") and a run-artifact source both fail. We find the
# real logged-model id and register/point the version at its true location instead.
run = client.get_run(run_id)
model_id = None
try:
    mo = run.outputs.model_outputs
    if mo:
        model_id = mo[0].model_id
except Exception as e:
    print("run.outputs unavailable:", e)

if model_id is None:
    try:
        lms = client.search_logged_models(experiment_ids=[exp.experiment_id])
        cand = [m for m in lms if getattr(m, "source_run_id", None) == run_id]
        if cand:
            model_id = cand[0].model_id
    except Exception as e:
        print("search_logged_models unavailable:", e)

assert model_id, "could not locate the logged model id for the champion run"
logged_model = client.get_logged_model(model_id)
print("logged model_id:", model_id, "| artifact_location:", logged_model.artifact_location)

# make sure the registered-model container exists (safe to rerun)
try:
    client.get_registered_model(MODEL_NAME)
except Exception:
    client.create_registered_model(MODEL_NAME)

# register a version pointing at the REAL artifact location
mv = client.create_model_version(name=MODEL_NAME, source=logged_model.artifact_location, run_id=run_id)
print(f"registered {MODEL_NAME} version {mv.version}")


## Load the champion back from the registry

This is the part the assignment asks for: the submission comes from a model pulled out of
the registry, not from an object still sitting in memory from training. We load the latest
version by number so we do not depend on stage transitions being set.

In [ ]:
# prefer loading straight from the model registry (what the assignment asks for), but fall
# back to the logged-model id and then the raw artifact location if the DagsHub registry
# resolver misbehaves. Whichever succeeds gives us the exact same pipeline.
candidates = [
    (f"models:/{MODEL_NAME}/{mv.version}", "registry version"),
    (f"models:/{model_id}",               "logged model id"),
    (logged_model.artifact_location,      "artifact location"),
]
champion = None
for uri, label in candidates:
    try:
        champion = mlflow.sklearn.load_model(uri)
        print(f"loaded from {label}: {uri}")
        break
    except Exception as e:
        print(f"could not load from {label} ({uri}) -> {type(e).__name__}: {e}")

assert champion is not None, "every load path failed; paste the printed errors back"
print("loaded pipeline:", type(champion).__name__)
print(champion)


## Predict on the raw test file and build the submission

The pipeline handles everything internally, so we hand it the untouched test frame. Kaggle
wants two columns, `Id` in the form `Store_Dept_Date` and `Weekly_Sales`. The Pipeline keeps
row order, so we build the ids straight from the test frame in the same order. Weekly sales
cannot really be negative, so we floor the few tiny negative predictions at zero.

In [ ]:
preds = champion.predict(test)   # raw, unprocessed test frame straight through
assert len(preds) == len(test)
assert np.isfinite(preds).all(), "predictions contain NaN/inf"

preds = np.clip(preds, 0, None)  # sales are non-negative; drop this line to keep raw output

submission = pd.DataFrame({
    "Id": (test["Store"].astype(str) + "_" +
           test["Dept"].astype(str) + "_" +
           test["Date"].dt.strftime("%Y-%m-%d")),
    "Weekly_Sales": preds,
})

# sanity: the ids must line up one to one with Kaggle's sample submission
sample = pd.read_csv(os.path.join(DATA_DIR, "sampleSubmission.csv"))
assert len(submission) == len(sample)
assert set(submission["Id"]) == set(sample["Id"]), "submission ids do not match the sample"

SUBMISSION_PATH = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
print("wrote", SUBMISSION_PATH, submission.shape)
submission.head()

## Send it to Kaggle

This uploads the file and scores it. The competition is closed for the leaderboard, but
late submissions still get a private-leaderboard WMAE back, which is the number we report.

In [ ]:
MESSAGE = "champion pipeline from MLflow registry"
subprocess.run([
    "kaggle", "competitions", "submit",
    "-c", KAGGLE_COMP,
    "-f", SUBMISSION_PATH,
    "-m", MESSAGE,
], check=True)

# show the score once Kaggle finishes grading (rerun this cell if it still says pending)
subprocess.run(["kaggle", "competitions", "submissions", "-c", KAGGLE_COMP], check=True)